In [ ]:
import joblib
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split

In [ ]:
# Loading the datasets

df = pd.read_csv("../Data/Cleaned/ChargingRecords_Feature_Engineered.csv")

# Create classfication target 

demand_thershold = df['Demand'].median()

df["DemandCategory"]= (
    df['Demand'] >= demand_thershold
).astype(int)

# Separate features and target
X = df.drop(columns=['Demand','DemandCategory'])
y= df['DemandCategory']

# Recreate the original test split
X_train,X_test,y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Load the trained model

model = joblib.load(
    "../Models/best_classification_model.pkl"
)

print("Model loaded successfully!")
print("Test data shape:", X_test.shape)
print("Demand threshold:", demand_thershold)


In [ ]:
# Sample test records for faster explanation
X_sample = X_test.sample(
    n=500,
    random_state=42
)

# Create SHAP explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

# Handle binary classification output
if isinstance(shap_values, list):
    shap_values = shap_values[1]
elif shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

print("SHAP values shape:", shap_values.shape)

In [ ]:
# SHAP summary plot
shap.summary_plot(
    shap_values,
    X_sample,
    max_display=15,
    show=False
)

plt.title("SHAP Summary - High Demand Classification")
plt.tight_layout()
plt.show()

In [ ]:
# Predict sampled sessions
sample_predictions = model.predict(X_sample)

# Select the first predicted High Demand session
high_demand_positions = np.flatnonzero(sample_predictions == 1)

if len(high_demand_positions) == 0:
    print("No High Demand predictions in the sample.")
else:
    i = high_demand_positions[0]

    # Get the baseline for the positive class
    base_value = np.asarray(explainer.expected_value).ravel()
    base_value = base_value[-1]

    # Create an individual explanation
    explanation = shap.Explanation(
        values=shap_values[i],
        base_values=base_value,
        data=X_sample.iloc[i].to_numpy(),
        feature_names=X_sample.columns.tolist()
    )

    print("Predicted class: High Demand")
    print(
        "High Demand probability:",
        round(model.predict_proba(X_sample.iloc[[i]])[0, 1], 4)
    )

    shap.plots.waterfall(explanation, max_display=12)
    plt.show()

In [ ]:
# Load the best XGBoost regression model

reg_model = joblib.load(
    "../Models/best_xgboost_regression.pkl"
)

# Get the exact features used during training
feature_names = reg_model.get_booster().feature_names

if feature_names is None:
    raise ValueError("Model feature names are unavailable. Use the original training columns.")

missing_columns = [
    column for column in feature_names
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing model features: {missing_columns}")

# Select only the original features, in training order
X_reg = df.loc[:, feature_names]

X_reg_sample = X_reg.sample(
    n=500,
    random_state=42
)

print("Model features:", len(feature_names))
print("Sample shape:", X_reg_sample.shape)

# Generate SHAP values
reg_explainer = shap.TreeExplainer(reg_model)
reg_shap_values = reg_explainer.shap_values(X_reg_sample)

print("SHAP values shape:", reg_shap_values.shape)

# Plot SHAP summary
shap.summary_plot(
    reg_shap_values,
    X_reg_sample,
    max_display=15,
    show=False
)

plt.title("SHAP Summary - XGBoost Regression")
plt.tight_layout()
plt.show()

In [ ]:
# Select one session from the SHAP sample
i = 0

session = X_reg_sample.iloc[[i]]
prediction = reg_model.predict(session)[0]

# Create an individual SHAP explanation
explanation = shap.Explanation(
    values=reg_shap_values[i],
    base_values=float(np.asarray(reg_explainer.expected_value).squeeze()),
    data=session.iloc[0].to_numpy(),
    feature_names=X_reg_sample.columns.tolist()
)

print(f"Predicted Demand: {prediction:.2f}")

shap.plots.waterfall(explanation, max_display=12, show=False)
plt.show()

In [ ]:
feature_names = reg_model.get_booster().feature_names

if not feature_names:
    raise ValueError("The model does not contain feature names.")

X_reg = df.loc[:, feature_names]
y_reg = df["Demand"]

# Recreate the original regression split
X_train_reg, X_test_reg, _, _ = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

# Use training records to build the LIME explainer
X_lime_train = X_train_reg.sample(
    n=min(5000, len(X_train_reg)),
    random_state=42
)

lime_explainer = LimeTabularExplainer(
    training_data=X_lime_train.to_numpy(),
    feature_names=feature_names,
    mode="regression",
    random_state=42
)

def predict_demand(data: np.ndarray) -> np.ndarray:
    features = pd.DataFrame(data, columns=feature_names)
    return reg_model.predict(features)

# Explain one test session
session = X_test_reg.iloc[0].to_numpy()

lime_explanation = lime_explainer.explain_instance(
    data_row=session,
    predict_fn=predict_demand,
    num_features=10,
    num_samples=5000
)

print("Predicted Demand:", round(predict_demand([session])[0], 2))
print("LIME local explanation:")
print(lime_explanation.as_list())

lime_explanation.as_pyplot_figure()
plt.title("LIME Explanation - XGBoost Regression")
plt.tight_layout()
plt.show()